# Thu thập Google Trends Daily Data (Incremental Save)

## Cách hoạt động
1. Chia date range thành **batch 1 tháng** → Google Trends trả daily
2. Mỗi batch lấy xong → **append ngay vào checkpoint CSV** (`*_batches.csv`)
3. Bị 429 giữa chừng → **chạy lại cell, nó tự skip tháng đã lấy**
4. Lấy đủ tất cả tháng → Cell 4 rescale theo monthly reference và export file final

## Output
- `data/raw/EldenRing_trend_daily.csv`
- `data/raw/ReadyOrNot_trend_daily.csv`

## Cell 1: Setup

In [ ]:
!pip install pytrends -q

import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = os.path.join('..', 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
print(f'Output directory: {os.path.abspath(RAW_DIR)}')

## Cell 2: Cấu hình

In [ ]:
GAMES = {
    'EldenRing': {
        'keyword': 'Elden Ring',
        'start_year': 2022, 'start_month': 2,
        'stop_year': 2026,  'stop_month': 3,
        'output_file': 'EldenRing_trend_daily.csv'
    },
    'ReadyOrNot': {
        'keyword': 'Ready Or Not',
        'start_year': 2021, 'start_month': 12,
        'stop_year': 2026,  'stop_month': 3,
        'output_file': 'ReadyOrNot_trend_daily.csv'
    }
}
print('Games configured:', list(GAMES.keys()))

## Cell 3: Incremental Fetch (có Resume)

- Mỗi tháng = 1 API request → lưu ngay vào `*_batches.csv`
- Bị 429 → chờ hết rate-limit (hoặc đổi VPN) → **chạy lại cell này**
- Cell tự skip tháng đã lấy, chỉ fetch tháng còn thiếu
- `WAIT_TIME = 60` giây giữa mỗi request (an toàn nhất)

In [ ]:
from pytrends.request import TrendReq
from dateutil.relativedelta import relativedelta
from datetime import datetime

WAIT_TIME = 60  # Giây chờ. Tăng lên 90-120 nếu vẫn bị 429.

def generate_monthly_batches(start_year, start_month, stop_year, stop_month):
    batches = []
    current = datetime(start_year, start_month, 1)
    end_limit = datetime(stop_year, stop_month, 28)
    while current <= end_limit:
        batch_end = current + relativedelta(months=1) - pd.Timedelta(days=1)
        if batch_end > end_limit:
            batch_end = end_limit
        batches.append((current, batch_end))
        current = current + relativedelta(months=1)
    return batches

def load_checkpoint(path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        df = pd.read_csv(path, parse_dates=['date'])
        done = set(df['batch_month'].unique())
        return df, done
    return pd.DataFrame(), set()

def append_checkpoint(path, df_batch):
    header = not os.path.exists(path) or os.path.getsize(path) == 0
    df_batch.to_csv(path, mode='a', header=header, index=False)

# ===== MAIN =====
pytrends = TrendReq(hl='en-US', tz=420)

for game_name, cfg in GAMES.items():
    keyword = cfg['keyword']
    ckpt_path = os.path.join(RAW_DIR, f'{game_name}_batches.csv')
    
    print(f"\n{'='*60}")
    print(f"  {game_name} | '{keyword}'")
    print(f"  Checkpoint: {ckpt_path}")
    print(f"{'='*60}")
    
    _, done_months = load_checkpoint(ckpt_path)
    batches = generate_monthly_batches(
        cfg['start_year'], cfg['start_month'],
        cfg['stop_year'], cfg['stop_month']
    )
    remaining = [b for b in batches if b[0].strftime('%Y-%m') not in done_months]
    print(f"  Tổng: {len(batches)} tháng | Đã có: {len(done_months)} | Còn: {len(remaining)}")
    
    if not remaining:
        print(f"  🎉 ĐÃ ĐỦ DATA! Chạy Cell 4 để rescale & export.")
        continue
    
    for idx, (b_start, b_end) in enumerate(remaining):
        month_key = b_start.strftime('%Y-%m')
        timeframe = f"{b_start.strftime('%Y-%m-%d')} {b_end.strftime('%Y-%m-%d')}"
        batch_num = batches.index((b_start, b_end)) + 1
        print(f"  [{batch_num}/{len(batches)}] {month_key} → {timeframe} ...", end='', flush=True)
        
        try:
            pytrends.build_payload([keyword], timeframe=timeframe, geo='')
            df_batch = pytrends.interest_over_time()
            
            if df_batch.empty:
                print(' ⚠️ empty')
                continue
            
            save_df = pd.DataFrame({
                'date': df_batch.index,
                'value': df_batch[keyword].values,
                'batch_month': month_key
            })
            
            # === LƯU NGAY ===
            append_checkpoint(ckpt_path, save_df)
            done_months.add(month_key)
            print(f' ✅ {len(save_df)} rows saved')
            
        except Exception as e:
            print(f' ❌ {e}')
            print(f'\n  ⛔ Dừng tại {month_key}.')
            print(f'     Đã lưu: {len(done_months)}/{len(batches)} tháng.')
            print(f'     → Chờ hết rate-limit (vài phút) rồi CHẠY LẠI cell này.')
            break
        
        if idx < len(remaining) - 1:
            print(f'     ⏳ Chờ {WAIT_TIME}s...', flush=True)
            time.sleep(WAIT_TIME)
    
    # Summary
    _, final_done = load_checkpoint(ckpt_path)
    if len(final_done) >= len(batches):
        print(f'\n  🎉 {game_name}: HOÀN THÀNH! → Chạy Cell 4 để rescale.')
    else:
        print(f'\n  ⏸️ {game_name}: {len(final_done)}/{len(batches)} tháng. Chạy lại cell.')

## Cell 4: Rescale & Export

Đọc checkpoint → rescale theo monthly reference (file `EldenRing.csv`, `ReadyOrNot.csv`) → export final CSV.

**Rescale logic**: Với mỗi tháng, daily batch có scale 0-100 riêng. Ta nhân mỗi ngày với `monthly_ref / 100`
để đưa tất cả về cùng 1 thang.

In [ ]:
# Monthly reference files (đã có sẵn trong repo)
MONTHLY_REF = {
    'EldenRing': {'file': 'EldenRing.csv', 'date_col': 'Time', 'val_col': 'Elden Ring'},
    'ReadyOrNot': {'file': 'ReadyOrNot.csv', 'date_col': 'Time', 'val_col': 'Ready Or Not'}
}

results = {}

for game_name, cfg in GAMES.items():
    ckpt_path = os.path.join(RAW_DIR, f'{game_name}_batches.csv')
    if not os.path.exists(ckpt_path):
        print(f'⚠️ {game_name}: No checkpoint file. Run Cell 3 first.')
        continue
    
    # Load checkpoint (raw daily batches)
    df_raw = pd.read_csv(ckpt_path, parse_dates=['date'])
    df_raw = df_raw.drop_duplicates(subset=['date']).sort_values('date')
    print(f'\n{game_name}: {len(df_raw)} raw daily rows')
    
    # Load monthly reference
    ref = MONTHLY_REF[game_name]
    ref_path = os.path.join(RAW_DIR, ref['file'])
    df_monthly = pd.read_csv(ref_path)
    df_monthly['month'] = pd.to_datetime(df_monthly[ref['date_col']]).dt.to_period('M')
    monthly_map = dict(zip(df_monthly['month'].astype(str), df_monthly[ref['val_col']]))
    
    # Rescale: daily_scaled = daily_raw * (monthly_ref / 100)
    df_raw['month'] = df_raw['date'].dt.to_period('M').astype(str)
    df_raw['monthly_ref'] = df_raw['month'].map(monthly_map)
    df_raw['monthly_ref'] = df_raw['monthly_ref'].ffill().bfill()
    
    df_raw['scaled'] = df_raw['value'] * (df_raw['monthly_ref'] / 100.0)
    
    # Export
    export = pd.DataFrame({
        'Date': df_raw['date'],
        'Trend_Value': df_raw['scaled']
    }).sort_values('Date').reset_index(drop=True)
    
    export['Trend_Value'] = export['Trend_Value'].interpolate('linear').fillna(0).clip(lower=0)
    
    out_path = os.path.join(RAW_DIR, cfg['output_file'])
    export.to_csv(out_path, index=False)
    results[game_name] = export
    
    print(f'  ✅ Saved: {out_path} ({len(export)} rows)')
    print(f'     Range: {export["Date"].min()} → {export["Date"].max()}')
    print(f'     Trend_Value: min={export["Trend_Value"].min():.2f}, max={export["Trend_Value"].max():.2f}')
    print(export.head())

## Cell 5: Validation Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for idx, (game_name, ref) in enumerate(MONTHLY_REF.items()):
    ax = axes[idx]
    
    # Monthly gốc
    mf = os.path.join(RAW_DIR, ref['file'])
    if os.path.exists(mf):
        dm = pd.read_csv(mf)
        dm[ref['date_col']] = pd.to_datetime(dm[ref['date_col']])
        ax.plot(dm[ref['date_col']], dm[ref['val_col']], 'ro-', label='Monthly (gốc)', markersize=5, lw=2)
    
    # Daily mới
    df = os.path.join(RAW_DIR, GAMES[game_name]['output_file'])
    if os.path.exists(df):
        dd = pd.read_csv(df, parse_dates=['Date'])
        ax.plot(dd['Date'], dd['Trend_Value'], 'b-', label='Daily (rescaled)', alpha=0.6, lw=0.8)
    
    ax.set_title(f'{game_name}: Monthly vs Daily', fontsize=14, fontweight='bold')
    ax.set_ylabel('Search Interest')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(os.path.join('..', 'reports', 'figures'), exist_ok=True)
plt.savefig(os.path.join('..', 'reports', 'figures', 'trends_daily_vs_monthly.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/figures/trends_daily_vs_monthly.png')

## Cell 6: Hướng dẫn tích hợp

Sửa `process_and_merge.py` hàm `main()`:
```python
games = {
    'EldenRing': { ..., 'trend': 'EldenRing_trend_daily.csv', ... },
    'ReadyOrNot': { ..., 'trend': 'ReadyOrNot_trend_daily.csv', ... }
}
```
Code merge **không cần sửa** — tự detect `Date` + `Trend_Value`.